# Example 16: post-processing ROI and exclusion-zone editor

Plots a saved scan so you can draw the ROI and the include/exclude shapes on it. The shapes
are saved to a `laguna.postprocess` config, and the notebook then turns every scan into a
ground-surface DEM. No hardware is involved.

* **ROI rectangle**: the area that is kept. It also fixes the DEM grid, so every DEM lines up.
* **Include polygons** (optional): if any exist, only points inside one of them are kept.
* **Exclude polygons/rectangles/circles**: removed *before* denoising and ground filtering, so
  an obstruction can't distort the surface around it. Their DEM cells are set to nodata
  rather than interpolated.

Needs `pip install 'laguna[postprocess]'` and LAStools (see
`docs/guides/postprocess-scans.md`). All coordinates are in mm. They are the scan's own
unless you set an experiment frame in step 1.

In [ ]:
%matplotlib widget
from pathlib import Path

import matplotlib.pyplot as plt

from laguna.postprocess import (Circle, Polygon, Rect, find_scans, load_postprocess_config,
                                run, save_postprocess_config)
from laguna.postprocess.editor import ShapeEditor
from laguna.postprocess.preview import show_dem

# Point these at your data. Start the config from examples/example_postprocess.yaml.
SCAN_DIR = Path("~/Desktop/topo_test").expanduser()
CONFIG = SCAN_DIR / "postprocess.yaml"

cfg = load_postprocess_config(CONFIG)
scans = find_scans([SCAN_DIR])
print(f"{len(scans)} scans in {SCAN_DIR}; outputs go to {cfg.out_path}")

## 1. Experiment frame (optional)
`cfg.frame` is a rigid transform from scanner coordinates to the experiment frame. It uses the
same keys as `frames.experiment` in the rig config: `rotation_deg` about Z, then `translation`
or `origin` (the scanner point that becomes (0, 0, 0)), or a 4x4 `matrix`. It is applied
before everything else. The editor draws in the frame's coordinates, and the ROI and shapes are
saved in them, so set the frame **before** drawing. Changing it later means redrawing the shapes.
A shift that is a multiple of 30 mm keeps LAStools' grids aligned with scanner coordinates.

In [ ]:
# cfg.frame = {"rotation_deg": 0, "origin": [330, -480, 0]}
# cfg.frame = None  # back to scanner coordinates
cfg.frame

## 2. Pick a scan to draw on
The shapes apply to every scan, so choose one with good coverage.

In [ ]:
scan = scans[-1]
scan.name

## 3. Draw shapes
Pick a mode, draw, then press **Add shape**. **Preview DEM** runs the full pipeline on this
scan with the current shapes (writing to `<output_dir>/_preview`) and plots the result below
the editor. **Save config** writes the YAML file, which drops its comments.

In [ ]:
editor = ShapeEditor(scan, cfg, CONFIG)
editor

Shapes can also be edited in code. Call `editor.redraw()` afterwards:

In [ ]:
# cfg.exclude.append(Circle(x=765, y=-100, r=75, label="obstruction"))
# cfg.exclude.append(Polygon([(400, 0), (450, 0), (450, 40)], label="bolt"))
# cfg.roi = Rect(320, -480, 752, 368)
# del cfg.exclude[0]
# editor.redraw()
cfg.exclude

## 4. Save and process every scan
Scans whose outputs already match the current settings are skipped. Changing a shape or a
parameter reprocesses all of them.

In [ ]:
save_postprocess_config(cfg, CONFIG)
results = run(scans, cfg)
print({s: sum(r.status == s for r in results) for s in ("ok", "skipped", "failed")})
for r in results:
    if r.status == "failed":
        print(r.scan, r.error)

## 5. Look at the DEMs

In [ ]:
picks = [scans[0], scans[len(scans) // 2], scans[-1]]
with plt.ioff():
    fig, axs = plt.subplots(1, len(picks), figsize=(12, 7), sharex=True, sharey=True)
for ax, s in zip(axs, picks):
    show_dem(cfg.out_path / "dem" / f"{s.stem}.tif", ax=ax, cfg=cfg)
fig.canvas.header_visible = False
fig.canvas